# Itchy Ablation Study

Test each trick individually against the v1 baseline.
Same architecture, same training, one change at a time.

**Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**

Expected time: ~25 min total (7 models x ~3 min each)

In [ ]:
!pip install -q torch numpy huggingface-hub sentencepiece
import torch, math, time, os, shutil, numpy as np
import torch.nn.functional as F
from torch import nn
from pathlib import Path
from huggingface_hub import hf_hub_download
print(f'GPU: {torch.cuda.get_device_name(0)}')
device = torch.device('cuda')

In [ ]:
# Data setup
import sentencepiece as spm
os.makedirs('data/bytes', exist_ok=True)

def dl(fn, sub, dst):
    if Path(dst).exists(): return
    c = str(Path(hf_hub_download(repo_id='willdepueoai/parameter-golf',
        filename=fn, subfolder=sub, repo_type='dataset')).resolve())
    try: os.link(c, dst)
    except: shutil.copy2(c, dst)

dl('fineweb_train_000000.bin','datasets/datasets/fineweb10B_sp1024','data/train.bin')
dl('fineweb_val_000000.bin','datasets/datasets/fineweb10B_sp1024','data/val.bin')
dl('fineweb_1024_bpe.model','datasets/tokenizers','data/tok.model')

def load_shard(p):
    h = np.fromfile(p, dtype='<i4', count=256)
    return np.fromfile(p, dtype='<u2', count=int(h[2]), offset=256*4)

def write_shard(p, d):
    h = np.zeros(256,dtype='<i4'); h[0]=20240520; h[1]=1; h[2]=len(d)
    with open(p,'wb') as f: f.write(h.tobytes()); f.write(d.astype('<u2').tobytes())

if not Path('data/bytes/train.bin').exists():
    sp = spm.SentencePieceProcessor(model_file='data/tok.model')
    for s,d in [('data/train.bin','data/bytes/train.bin'),('data/val.bin','data/bytes/val.bin')]:
        print(f'Converting {s}...')
        tok = load_shard(s)
        bs = [np.frombuffer(sp.decode(tok[i:i+100000].tolist()).encode('utf-8'),dtype=np.uint8)
              for i in range(0,len(tok),100000)]
        write_shard(d, np.concatenate(bs))

def load_bytes(p):
    h = np.fromfile(p,dtype='<i4',count=256)
    return np.fromfile(p,dtype='<u2',count=int(h[2]),offset=256*4).astype(np.int64)

train_data = load_bytes('data/bytes/train.bin')
val_data = load_bytes('data/bytes/val.bin')
print(f'Train: {len(train_data):,} | Val: {len(val_data):,}')

In [ ]:
# ============================================================
# BUILDING BLOCKS
# ============================================================

class Rotary(nn.Module):
    def __init__(self, dim, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        self.register_buffer('inv_freq', inv_freq, persistent=False)
        self._cache = None
    def forward(self, seq_len, device, dtype):
        if self._cache is None or self._cache[0] != seq_len:
            t = torch.arange(seq_len, device=device, dtype=self.inv_freq.dtype)
            freqs = torch.outer(t, self.inv_freq.to(device))
            self._cache = (seq_len, freqs.cos()[None,None,:,:], freqs.sin()[None,None,:,:])
        return self._cache[1].to(dtype), self._cache[2].to(dtype)

def apply_rope(x, cos, sin, rope_dims=0):
    if rope_dims > 0 and rope_dims < x.size(-1):
        xr, xp = x[..., :rope_dims], x[..., rope_dims:]
        h = rope_dims // 2
        x1, x2 = xr[..., :h], xr[..., h:]
        xr = torch.cat((x1*cos+x2*sin, x1*(-sin)+x2*cos), dim=-1)
        return torch.cat((xr, xp), dim=-1)
    h = x.size(-1) // 2
    x1, x2 = x[..., :h], x[..., h:]
    return torch.cat((x1*cos+x2*sin, x1*(-sin)+x2*cos), dim=-1)

class Attn(nn.Module):
    def __init__(self, dim, nh, nkv, rope_base, qk_gain, rope_dims=0):
        super().__init__()
        self.nh, self.nkv = nh, nkv
        self.hd = dim // nh
        self.rd = rope_dims if rope_dims > 0 else self.hd
        kv = nkv * self.hd
        self.cq = nn.Linear(dim, dim, bias=False)
        self.ck = nn.Linear(dim, kv, bias=False)
        self.cv = nn.Linear(dim, kv, bias=False)
        self.proj = nn.Linear(dim, dim, bias=False)
        nn.init.zeros_(self.proj.weight)
        self.qg = nn.Parameter(torch.full((nh,), qk_gain))
        self.rot = Rotary(self.rd, rope_base)
    def forward(self, x):
        B,S,D = x.shape
        q = self.cq(x).reshape(B,S,self.nh,self.hd).transpose(1,2)
        k = self.ck(x).reshape(B,S,self.nkv,self.hd).transpose(1,2)
        v = self.cv(x).reshape(B,S,self.nkv,self.hd).transpose(1,2)
        q, k = F.rms_norm(q,(q.size(-1),)), F.rms_norm(k,(k.size(-1),))
        cos, sin = self.rot(S, x.device, q.dtype)
        q = apply_rope(q, cos, sin, self.rd if self.rd < self.hd else 0)
        k = apply_rope(k, cos, sin, self.rd if self.rd < self.hd else 0)
        q = q * self.qg.to(q.dtype)[None,:,None,None]
        y = F.scaled_dot_product_attention(q,k,v,is_causal=True,
            enable_gqa=(self.nkv!=self.nh))
        return self.proj(y.transpose(1,2).reshape(B,S,D))

class MLP_relu2(nn.Module):
    def __init__(self, dim, mult):
        super().__init__()
        self.fc = nn.Linear(dim, dim*mult, bias=False)
        self.proj = nn.Linear(dim*mult, dim, bias=False)
        nn.init.zeros_(self.proj.weight)
    def forward(self, x):
        return self.proj(torch.relu(self.fc(x)).square())

class MLP_leaky(nn.Module):
    def __init__(self, dim, mult):
        super().__init__()
        self.fc = nn.Linear(dim, dim*mult, bias=False)
        self.proj = nn.Linear(dim*mult, dim, bias=False)
        nn.init.zeros_(self.proj.weight)
    def forward(self, x):
        return self.proj(F.leaky_relu(self.fc(x), 0.5).square())

class ByteNgramHash(nn.Module):
    def __init__(self, hvocab, edim, mdim, sizes=(2,3,4,5,6)):
        super().__init__()
        self.hv, self.sizes = hvocab, sizes
        self.embed = nn.Embedding(hvocab * len(sizes), edim)
        nn.init.zeros_(self.embed.weight)
        self.proj = nn.Linear(edim, mdim, bias=False) if edim != mdim else None
        if self.proj: nn.init.zeros_(self.proj.weight)
        self.scale = nn.Parameter(torch.tensor(0.05))
        self.ps = [36313,27191,51637,39371,73291]
    def forward(self, ids):
        t = ids.int(); B,S = t.shape
        r = torch.zeros(B,S,self.embed.embedding_dim, device=t.device)
        for idx,n in enumerate(self.sizes):
            off = idx * self.hv
            h = torch.zeros_like(t)
            for k in range(n):
                p = self.ps[k%len(self.ps)]
                if k==0: h = p*t
                else: h = h ^ (p * F.pad(t[:,:-k],(k,0)))
            ix = (h.abs() % self.hv + off).long()
            if n>1:
                m = torch.cat([torch.zeros(B,n-1,device=t.device),torch.ones(B,S-n+1,device=t.device)],1).long()
                ix = ix*m + off*(1-m)
            r = r + self.embed(ix)
        if self.proj: r = self.proj(r)
        return r * self.scale

class Block(nn.Module):
    def __init__(self, dim, nh, nkv, mlp_mult, rope_base, qk_gain,
                 rope_dims=0, layer_idx=0, ln_scale=False, leaky=False):
        super().__init__()
        self.attn = Attn(dim, nh, nkv, rope_base, qk_gain, rope_dims)
        self.mlp = MLP_leaky(dim, mlp_mult) if leaky else MLP_relu2(dim, mlp_mult)
        self.as_ = nn.Parameter(torch.ones(dim))
        self.ms = nn.Parameter(torch.ones(dim))
        self.rm = nn.Parameter(torch.stack((torch.ones(dim), torch.zeros(dim))))
        self.lnf = 1.0/math.sqrt(layer_idx+1) if ln_scale else 1.0
    def forward(self, x, x0):
        m = self.rm.to(x.dtype)
        x = m[0][None,None,:]*x + m[1][None,None,:]*x0
        x = x + self.as_.to(x.dtype)[None,None,:] * self.attn(F.rms_norm(x,(x.size(-1),))*self.lnf)
        x = x + self.ms.to(x.dtype)[None,None,:] * self.mlp(F.rms_norm(x,(x.size(-1),))*self.lnf)
        return x

class ItchyAblation(nn.Module):
    def __init__(self, dim=256, num_layers=8, nh=8, nkv=4, mlp_mult=2,
                 patch_size=4, rope_dims=0, ln_scale=False, leaky=False,
                 ngram_vocab=0, ngram_dim=64, softcap=30.0):
        super().__init__()
        self.dim, self.ps, self.sc = dim, patch_size, softcap
        self.be = nn.Embedding(260, dim)
        self.pp = nn.Linear(dim*patch_size, dim, bias=False)
        self.ng = ByteNgramHash(ngram_vocab, ngram_dim, dim) if ngram_vocab > 0 else None
        self.blocks = nn.ModuleList([
            Block(dim, nh, nkv, mlp_mult, 10000.0, 1.5,
                  rope_dims=rope_dims, layer_idx=i, ln_scale=ln_scale, leaky=leaky)
            for i in range(num_layers)])
        self.up = nn.Linear(dim, patch_size*260, bias=False)
        ne = num_layers//2; nd = num_layers-ne
        self.ne, self.nd = ne, nd
        self.sw = nn.Parameter(torch.ones(min(ne,nd), dim))

    def forward(self, ids, tgt):
        B,S = ids.shape; P = self.ps
        x = self.be(ids).reshape(B,S//P,P*self.dim)
        x = F.rms_norm(self.pp(x),(self.dim,))
        if self.ng is not None:
            ng = self.ng(ids).reshape(B,S//P,P,-1).mean(2)
            x = x + ng.to(x.dtype)
        x0 = x; skips = []
        for i in range(self.ne): x = self.blocks[i](x,x0); skips.append(x)
        for i in range(self.nd):
            if skips: x = x + self.sw[i].to(x.dtype)[None,None,:]*skips.pop()
            x = self.blocks[self.ne+i](x,x0)
        x = F.rms_norm(x,(x.size(-1),))
        lo = self.up(x).reshape(-1,260)
        lo = self.sc * torch.tanh(lo/self.sc)
        return F.cross_entropy(lo.float(), tgt.reshape(-1))

n_params = lambda m: sum(p.numel() for p in m.parameters())
print('Building blocks defined')

In [ ]:
# ============================================================
# TRAINING + VALIDATION HARNESS
# ============================================================

SEQ_LEN = 512
BATCH = 8192
STEPS = 3000
LR = 3e-4

def train_and_val(name, model):
    model = model.to(device).bfloat16()
    p = n_params(model)
    print(f'\n{"="*60}')
    print(f'{name}: {p:,} params ({p/1e6:.1f}M)')
    print(f'{"="*60}')
    opt = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9,0.95), weight_decay=0.01)
    pos = 0; t0 = time.time(); losses = []
    for step in range(1, STEPS+1):
        usable = (BATCH//SEQ_LEN)*SEQ_LEN
        end = pos+usable+1
        if end > len(train_data): pos=0; end=usable+1
        chunk = train_data[pos:end]; pos = end-1
        x = torch.tensor(chunk[:-1].reshape(-1,SEQ_LEN), device=device)
        y = torch.tensor(chunk[1:].reshape(-1,SEQ_LEN), device=device)
        opt.zero_grad()
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            loss = model(x, y)
        loss.backward()
        opt.step()
        losses.append(loss.item())
        if step % 1000 == 0:
            avg = np.mean(losses[-1000:])
            print(f'  step {step:>5} | bpb {avg/math.log(2):.4f} | {time.time()-t0:.0f}s')
    train_time = time.time() - t0
    # Validation
    vl = []
    for i in range(0, min(200*SEQ_LEN, len(val_data)-SEQ_LEN-1), SEQ_LEN):
        chunk = val_data[i:i+SEQ_LEN+1]
        x = torch.tensor(chunk[:-1].reshape(1,-1), device=device)
        y = torch.tensor(chunk[1:].reshape(1,-1), device=device)
        with torch.no_grad():
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                vl.append(model(x,y).item())
    val = np.mean(vl)
    bpb = val / math.log(2)
    print(f'  >>> VAL BPB: {bpb:.4f} | {p/1e6:.1f}M params | {train_time:.0f}s')
    return {'name': name, 'params': p, 'val_bpb': bpb, 'time': train_time}

print('Harness ready')

In [ ]:
# ============================================================
# RUN ALL ABLATIONS
# ============================================================

results = []

# A. BASELINE: relu2, full RoPE, 2x MLP, no extras
torch.manual_seed(42)
results.append(train_and_val('A. Baseline',
    ItchyAblation(dim=256, num_layers=8, mlp_mult=2)))

# B. LeakyReLU(0.5)2 only
torch.manual_seed(42)
results.append(train_and_val('B. +LeakyReLU(0.5)^2',
    ItchyAblation(dim=256, num_layers=8, mlp_mult=2, leaky=True)))

# C. 3x MLP only (more params in MLP)
torch.manual_seed(42)
results.append(train_and_val('C. +3x MLP',
    ItchyAblation(dim=256, num_layers=8, mlp_mult=3)))

# D. Partial RoPE only (8 of 32 head dims)
torch.manual_seed(42)
results.append(train_and_val('D. +Partial RoPE 8/32',
    ItchyAblation(dim=256, num_layers=8, mlp_mult=2, rope_dims=8)))

# E. LN Scale only
torch.manual_seed(42)
results.append(train_and_val('E. +LN Scale',
    ItchyAblation(dim=256, num_layers=8, mlp_mult=2, ln_scale=True)))

# F. N-gram hash only (adds ~1.3M params)
torch.manual_seed(42)
results.append(train_and_val('F. +Ngram hash 2048',
    ItchyAblation(dim=256, num_layers=8, mlp_mult=2, ngram_vocab=2048)))

# G. Best combo (whatever looks good from above)
torch.manual_seed(42)
results.append(train_and_val('G. +Leaky +3x MLP',
    ItchyAblation(dim=256, num_layers=8, mlp_mult=3, leaky=True)))

In [ ]:
# ============================================================
# RESULTS TABLE
# ============================================================

print()
print('='*70)
print('ABLATION RESULTS')
print('='*70)
print(f'{"Config":<30} {"Params":>8} {"Val BPB":>9} {"Delta":>9} {"Time":>6}')
print('-'*70)

base_bpb = results[0]['val_bpb']
best_bpb = min(r['val_bpb'] for r in results)
for r in results:
    d = r['val_bpb'] - base_bpb
    tag = ''
    if d == 0: tag = ' <-- baseline'
    elif r['val_bpb'] == best_bpb: tag = ' *** BEST'
    elif d < 0: tag = ' +'
    elif d > 0: tag = ' -'
    print(f'{r["name"]:<30} {r["params"]/1e6:>7.1f}M {r["val_bpb"]:>9.4f} {d:>+9.4f} {r["time"]:>5.0f}s{tag}')

print()
b = min(results, key=lambda r: r['val_bpb'])
print(f'Best: {b["name"]} at {b["val_bpb"]:.4f} BPB ({base_bpb - b["val_bpb"]:+.4f} vs baseline)')